# 🤖 Üretken Yapay Zeka — Ders 2
## Üretken Modellerin Türetilmesi
### MAP · MLE · Beta-Binom · Dirichlet

**Haydar Kılıç | Mühendislik Fakültesi, Yapay Zeka Mühendisliği**

---
Bu notebook, ders slaytlarındaki teorik kavramların Python ile uygulamalı gösterimini içermektedir.

In [ ]:
# [H5] comb kaldırıldı — kullanılmıyor
# [H6] dirichlet (scipy.stats) kaldırıldı — np.random.dirichlet kullanılıyor
# [H7] mpatches kaldırıldı — hiçbir yerde kullanılmıyor
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist
from scipy.stats import binom
from scipy.special import betaln, gammaln
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# [H1] np.trapz NumPy 2.0'da kaldırıldı → np.trapezoid kullan
# Geriye dönük uyumluluk için sarmalayıcı:
if not hasattr(np, 'trapezoid'):
    np.trapezoid = np.trapz  # NumPy < 2.0 desteği

plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

NAVY = '#1a237e'
BLUE = '#1565c0'
print('✅ Kütüphaneler yüklendi.')

---
## 📌 BÖLÜM 1: Pozitif Örneklerden Öğrenme & Number Game

### 1.1 Kavram Öğrenme = İkili Sınıflandırma

Bir çocuğun kavramları **yalnızca pozitif örneklerden** öğrendiği gibi, modelimiz de  
D = {x₁, …, xₙ} pozitif örneklerinden hangi sayıların C kümesine ait olduğunu öğrenir.

**Sonsal Öngörüsel Dağılım:** $p(\tilde{x} \in C \mid \mathcal{D})$ — yeni bir sayının C kapsamında olma olasılığı.

In [ ]:
# ── Sayı Oyunu: Hipotez Uzayının Tanımlanması ─────────────────────────────────
UNIVERSE = set(range(1, 101))   # {1, …, 100}

def build_hypotheses():
    """1-100 arasında tanımlı aritmetik hipotezler."""
    H = {}
    H['tek sayılar']  = {x for x in UNIVERSE if x % 2 != 0}
    H['çift sayılar'] = {x for x in UNIVERSE if x % 2 == 0}
    H['kareler']      = {x for x in UNIVERSE if int(x**0.5)**2 == x}
    for k in range(3, 11):
        H[f'{k} katları'] = {x for x in UNIVERSE if x % k == 0}
    for j in range(10):
        H[f'{j} ile bitenler'] = {x for x in UNIVERSE if x % 10 == j}
    for b in range(2, 11):
        H[f'{b} nin kuvvetleri'] = set()
        v = b
        while v <= 100:
            H[f'{b} nin kuvvetleri'].add(v)
            v *= b
    H['tümü'] = set(UNIVERSE)
    # Aralıklar (10'ar birimlik)
    for lo in range(1, 100, 10):
        hi = min(lo + 9, 100)
        H[f'[{lo},{hi}] aralığı'] = {x for x in UNIVERSE if lo <= x <= hi}
    return H

hypotheses = build_hypotheses()

print(f'📐 Toplam hipotez sayısı: {len(hypotheses)}')
print('\nBazı hipotezler:')
for name in ['2 nin kuvvetleri', 'çift sayılar', 'kareler', 'tek sayılar']:
    s = sorted(hypotheses[name])
    print(f'  {name:25s}: {s[:10]} … |h|={len(s)}')

---
### 1.2 Güçlü Örnekleme Varsayımı & Boyut Prensibi (Size Principle)

N pozitif örneğin h hipotezinden **tekdüze** örneklendiğini varsayarsak:

$$p(\mathcal{D} \mid h) = \left(\frac{1}{|h|}\right)^N$$

**Boyut Prensibi:** Dar kapsamlı hipotez (küçük |h|) → daha yüksek olabilirlik.

In [ ]:
# ── Boyut Prensibinin Sayısal Gösterimi ───────────────────────────────────────
def likelihood(data, h_set):
    """p(D|h) = (1/|h|)^N  eğer D ⊆ h, aksi 0."""
    if not set(data).issubset(h_set):
        return 0.0
    return (1.0 / len(h_set)) ** len(data)

# Örnek: D = {8} için iki rakip hipotezi karşılaştır
D_example = [8]
h1 = hypotheses['2 nin kuvvetleri']   # dar:  {2,4,8,16,32,64} -> |h|=6
h2 = hypotheses['çift sayılar']       # geniş: 50 eleman

L1 = likelihood(D_example, h1)
L2 = likelihood(D_example, h2)

print('🎯 BOYUT PRENSİBİ — D = {8} için Olabilirlik Karşılaştırması')
print('=' * 58)
print(f"h₁ = '2'nin kuvvetleri'  |h₁| = {len(h1):3d}  p(D|h₁) = {L1:.6f}")
print(f"h₂ = 'çift sayılar'      |h₂| = {len(h2):3d}  p(D|h₂) = {L2:.6f}")
print(f"\n  Olabilirlik oranı L1/L2 = {L1/L2:.2f}x")
print(f"  → h₁ veriyi h₂'ye göre {L1/L2:.1f} kat daha iyi açıklıyor!")

# N arttıkça fark nasıl açılır?
fig, ax = plt.subplots(figsize=(10, 5))
N_vals = np.arange(1, 10)
ratio  = [(len(h2)/len(h1))**n for n in N_vals]
ax.semilogy(N_vals, ratio, 'bo-', lw=2.5, ms=8)
ax.set_xlabel('Gözlem Sayısı N', fontsize=13)
ax.set_ylabel('Olabilirlik Oranı p(D|h₁) / p(D|h₂)  [log ölçek]', fontsize=12)
ax.set_title('Boyut Prensibi: N arttıkça dar hipotez baskın hale gelir', fontsize=13, fontweight='bold')
ax.set_xticks(N_vals)
for n, r in zip(N_vals, ratio):
    ax.annotate(f'{r:.0f}x', (n, r), textcoords='offset points', xytext=(4, 4), fontsize=9)
plt.tight_layout()
plt.show()

---
### 1.3 Önsel, Olabilirlik, Sonsal — Bayesçi Güncelleme

$$p(h \mid \mathcal{D}) = \frac{p(\mathcal{D}\mid h)\, p(h)}{\sum_{h'} p(\mathcal{D}\mid h')\, p(h')}$$

In [ ]:
# ── Önsel Tanımı ──────────────────────────────────────────────────────────────
def build_prior(hypotheses, pi0=0.6):
    """Karışım önseli: π₀·p_rules + (1-π₀)·p_interval."""
    rule_keys     = [k for k in hypotheses if 'aralığı' not in k]
    interval_keys = [k for k in hypotheses if 'aralığı' in  k]
    prior = {}
    for k in rule_keys:
        prior[k] = pi0 / len(rule_keys)
    for k in interval_keys:
        prior[k] = (1 - pi0) / len(interval_keys)
    return prior

def compute_posterior(data, hypotheses, prior):
    """Bayes kuralıyla sonsal olasılıkları hesapla."""
    unnorm = {}
    for h, h_set in hypotheses.items():
        unnorm[h] = likelihood(data, h_set) * prior.get(h, 0.0)
    Z = sum(unnorm.values())
    if Z == 0:
        return {h: 0.0 for h in hypotheses}
    return {h: v / Z for h, v in unnorm.items()}

def posterior_predictive(x_tilde, posterior, hypotheses):
    """BMA: p(x̃ ∈ C | D) = Σ_h p(y=1|x̃,h)·p(h|D)"""
    prob = 0.0
    for h, p_h in posterior.items():
        prob += (1.0 if x_tilde in hypotheses[h] else 0.0) * p_h
    return prob

# ── Görselleştirme: D={16} ve D={16,8,2,64} ──────────────────────────────────
prior = build_prior(hypotheses, pi0=0.7)

scenarios = [
    {'data': [16],          'title': 'D = {16}',
     'color': '#42A5F5'},
    {'data': [16, 8, 2, 64],'title': 'D = {16, 8, 2, 64}',
     'color': '#1565c0'},
    {'data': [16,23,19,20], 'title': 'D = {16, 23, 19, 20}',
     'color': '#7B1FA2'},
]

fig, axes = plt.subplots(len(scenarios), 1, figsize=(13, 11))
fig.suptitle('Number Game — Sonsal Öngörüsel Dağılım p(x̃ ∈ C | D)',
             fontsize=14, fontweight='bold')

xs = np.arange(1, 101)
for ax, sc in zip(axes, scenarios):
    post = compute_posterior(sc['data'], hypotheses, prior)
    preds = np.array([posterior_predictive(x, post, hypotheses) for x in xs])
    preds /= (preds.max() + 1e-12)   # normalize to [0,1] for display

    ax.bar(xs, preds, color=sc['color'], alpha=0.85, width=0.8)
    for d in sc['data']:
        ax.axvline(x=d, color='red', lw=1.5, alpha=0.7)
    ax.set_title(sc['title'] + '  → pozitif örnekler kırmızı çizgi ile işaretli',
                 fontsize=12, fontweight='bold')
    ax.set_xlim(0, 101)
    ax.set_ylim(0, 1.15)
    ax.set_xticks(range(0, 101, 4))
    ax.set_ylabel('p(x̃ ∈ C|D)', fontsize=10)

axes[-1].set_xlabel('x̃', fontsize=12)
plt.tight_layout()
plt.show()

---
### 1.4 MAP Kestirimi ve N → ∞ Davranışı

Yeterli veriyle sonsal tek bir hipotez üzerinde yoğunlaşır (Dirac ölçüsü):

$$p(h \mid \mathcal{D}) \to \delta_{\hat{h}_{MAP}}(h), \qquad \hat{h}_{MAP} = \arg\max_h\, p(h\mid\mathcal{D})$$

**MLE:** Veri çok olduğunda önsel gölgelenir → MAP → MLE

In [ ]:
# ── N arttıkça MAP'ın keskinleşmesi ──────────────────────────────────────────
true_concept = hypotheses['2 nin kuvvetleri']   # gerçek gizli kavram
true_samples = sorted(true_concept)              # örneklenebilecek sayılar

np.random.seed(42)
sample_sizes = [1, 2, 4, 8]

# Görselleştirme: üst 8 hipotezin sonsal olasılığı vs N
key_hyps = ['2 nin kuvvetleri', 'çift sayılar', '4 nin kuvvetleri',
            'kareler', '4 katları', '8 katları', 'tümü', 'tek sayılar']
colors_k  = ['#D32F2F','#1565c0','#F57C00','#388E3C',
              '#7B1FA2','#00838F','#455A64','#BF360C']

fig, axes = plt.subplots(1, len(sample_sizes), figsize=(16, 5), sharey=True)
fig.suptitle('MAP Yakınsaması: N arttıkça Sonsal Keskinleşir\n(Gerçek kavram: "2\'nin kuvvetleri")',
             fontsize=13, fontweight='bold')

for ax, N in zip(axes, sample_sizes):
    data = list(np.random.choice(true_samples, size=N, replace=True))
    post = compute_posterior(data, hypotheses, prior)

    vals  = [post.get(h, 0) for h in key_hyps]
    bars  = ax.barh(range(len(key_hyps)), vals,
                    color=colors_k, alpha=0.85, edgecolor='white')
    ax.set_yticks(range(len(key_hyps)))
    # [H3] DÜZELTİLDİ: ' nin ' → "'nin " (etiket bozulması giderildi)
    ax.set_yticklabels([h.replace(' nin ', "'nin ") for h in key_hyps], fontsize=9)
    ax.set_title(f'N = {N}\nD = {sorted(set(data))}', fontsize=10)
    ax.set_xlabel('p(h|D)')
    ax.set_xlim(0, 1.0)

plt.tight_layout()
plt.show()

# MAP vs MLE karşılaştırması (sözel)
# [H2] DÜZELTİLDİ: f-string dış sarmalayıcı tek tırnak → çift tırnak
print('\n📊 MAP vs MLE Karşılaştırması:')
print('─'*52)
print(f"  {'N':>4}  {'MAP hipotezi':30}  {'p(h|D)'}")
print('─'*52)
for N in [1, 2, 4, 8, 16]:
    data = list(np.random.choice(true_samples, size=N, replace=True))
    post = compute_posterior(data, hypotheses, prior)
    map_h  = max(post, key=post.get)
    map_p  = post[map_h]
    print(f'  {N:>4}  {map_h:30}  {map_p:.4f}')

---
### 1.5 Bayes Model Ortalaması (BMA) vs Tak-Çıkar Yaklaşımı

| Yöntem | Formül | Avantaj | Dezavantaj |
|--------|--------|---------|------------|
| **BMA** | $p(\tilde{x}\|D)=\sum_h p(\tilde{x}\|h)p(h\|D)$ | Belirsizliği korur | Hesaplama maliyetli |
| **Tak-Çıkar** | $p(\tilde{x}\|D)=p(\tilde{x}\|\hat{h})$ | Hızlı ve basit | Belirsizlik kaybolur |

In [ ]:
# ── BMA vs Tak-Çıkar ──────────────────────────────────────────────────────────
data_bma = [16, 8, 2, 64]
post_bma  = compute_posterior(data_bma, hypotheses, prior)
map_h_bma = max(post_bma, key=post_bma.get)

# BMA tahmini
bma_preds    = np.array([posterior_predictive(x, post_bma, hypotheses) for x in xs])

# Tak-çıkar tahmini (sadece MAP hipotezini kullan)
plugin_preds = np.array([1.0 if x in hypotheses[map_h_bma] else 0.0 for x in xs])

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
fig.suptitle(f'BMA vs Tak-Çıkar Yaklaşımı  (D = {data_bma})', fontsize=14, fontweight='bold')

for ax, preds, title, color in zip(
        axes,
        [bma_preds / bma_preds.max(), plugin_preds],
        [f'BMA — Tüm hipotezlerin ağırlıklı ortalaması',
         f'Tak-Çıkar — Yalnızca MAP: "{map_h_bma}" (p={post_bma[map_h_bma]:.3f})'],
        ['#1565c0', '#C62828']):
    ax.bar(xs, preds, color=color, alpha=0.8, width=0.8)
    for d in data_bma:
        ax.axvline(x=d, color='gold', lw=2, alpha=0.9, label='Eğitim verisi' if d == data_bma[0] else '')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(0, 101); ax.set_ylim(0, 1.2)
    ax.set_xticks(range(0, 101, 4))
    ax.set_ylabel('p(x̃ ∈ C|D)')
    ax.legend()

axes[-1].set_xlabel('x̃')
plt.tight_layout()
plt.show()

print('💡 BMA belirsizliği dağılıma yayarken, Tak-Çıkar tüm kütleyi')
print(f'   MAP hipotezi "{map_h_bma}" üzerine yığar.')

---
## 📌 BÖLÜM 2: Parametrik Bayes — Beta-Binom Modeli
### 2.1 Bernoulli Olabilirliği ve Yeterli İstatistikler

$X_i \sim \text{Ber}(\theta)$ için:

$$p(\mathcal{D}\mid\theta) = \theta^{N_1}(1-\theta)^{N_0}$$

$(N_1, N_0)$ — **yeterli istatistikler**: θ hakkında bilmemiz gereken tek şey.

In [ ]:
# ── Bernoulli Olabilirlik Yüzeyi ──────────────────────────────────────────────
theta_vals = np.linspace(0.001, 0.999, 500)

scenarios_bern = [
    {'N1': 3, 'N0': 0, 'label': 'N₁=3, N₀=0 (3 yazı)', 'color': '#C62828'},
    {'N1': 7, 'N0': 3, 'label': 'N₁=7, N₀=3 (7Y 3T)',  'color': '#1565c0'},
    {'N1': 5, 'N0': 5, 'label': 'N₁=5, N₀=5 (dengeli)', 'color': '#2E7D32'},
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Bernoulli Olabilirlik Fonksiyonu  p(D|θ) = θᴺ¹(1-θ)ᴺ⁰',
             fontsize=14, fontweight='bold')

for ax, sc in zip(axes, scenarios_bern):
    L = theta_vals ** sc['N1'] * (1 - theta_vals) ** sc['N0']
    L /= L.max()    # normalize
    mle = sc['N1'] / (sc['N1'] + sc['N0'])

    ax.plot(theta_vals, L, color=sc['color'], lw=2.5)
    ax.axvline(x=mle, color='black', ls='--', lw=2,
               label=f'MLE θ̂ = {mle:.2f}')
    ax.fill_between(theta_vals, L, alpha=0.15, color=sc['color'])
    ax.set_title(sc['label'], fontsize=11)
    ax.set_xlabel('θ'); ax.set_ylabel('p(D|θ)  [normalize]')
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print('📌 Yeterli İstatistik Özeti:')
print('   Tüm bilgi sadece (N₁, N₀) çiftinde saklı!')
print('   Verinin sırası önemli değil: YYTYYT ≡ YYYYTT vb.')

---
### 2.2 Beta Dağılımı — Konjuge Önsel

$$\text{Beta}(\theta \mid a, b) \propto \theta^{a-1}(1-\theta)^{b-1}$$

- **a** = sözde yazı sayısı, **b** = sözde tura sayısı (hiper-parametreler)
- Konjuge: Beta önsel × Bernoulli olabilirlik → **Beta sonsal**
- $\mathbb{E}[\theta] = \dfrac{a}{a+b}$

In [ ]:
# ── Beta Dağılımı: Farklı (a,b) parametreleri ─────────────────────────────────
theta_plot = np.linspace(0.001, 0.999, 500)

params = [
    (1,  1,  '#78909C', 'Beta(1,1) — Düzgün önsel'),
    (2,  2,  '#42A5F5', 'Beta(2,2) — Hafif orta eğilim'),
    (5,  2,  '#EF5350', 'Beta(5,2) — Yazıya eğilim (a>b)'),
    (2,  5,  '#66BB6A', 'Beta(2,5) — Turaya eğilim (b>a)'),
    (10, 10, '#AB47BC', 'Beta(10,10) — Güçlü orta önsel'),
    (0.5,0.5,'#FF7043', 'Beta(0.5,0.5) — U şekli (Jeffreys)'),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Beta Dağılımı: Farklı Hiper-parametreler (a, b)', fontsize=14, fontweight='bold')

for ax, (a, b, color, label) in zip(axes.flat, params):
    pdf = beta_dist.pdf(theta_plot, a, b)
    mean = a / (a + b)
    mode = (a - 1) / (a + b - 2) if (a > 1 and b > 1) else None

    ax.plot(theta_plot, pdf, color=color, lw=2.5)
    ax.fill_between(theta_plot, pdf, alpha=0.2, color=color)
    ax.axvline(x=mean, color='blue',  ls='--', lw=1.5, label=f'Ort. = {mean:.2f}')
    if mode is not None:
        ax.axvline(x=mode, color='red', ls=':', lw=1.5, label=f'Mod = {mode:.2f}')
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('θ'); ax.set_ylabel('p(θ)')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

---
### 2.3 Bayesçi Güncelleme: Beta Önsel + Bernoulli → Beta Sonsal

$$p(\theta \mid \mathcal{D}) = \text{Beta}(\theta \mid N_1 + a,\ N_0 + b)$$

Üsler toplanır — son derece zarif bir güncelleme kuralı!

In [ ]:
# ── Madeni Para Atışı: Ardışık Bayesçi Güncelleme ─────────────────────────────
np.random.seed(7)
theta_true = 0.7   # gerçek yazı olasılığı (bilinmiyor)
N_total    = 20
flips      = np.random.binomial(1, theta_true, N_total)  # 1=yazı 0=tura

a0, b0 = 2, 2      # önsel parametreler
checkpoints = [0, 1, 3, 5, 10, 20]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(f'Beta-Binom Bayesçi Güncelleme  (gerçek θ={theta_true}, önsel Beta({a0},{b0}))',
             fontsize=13, fontweight='bold')

for ax, n in zip(axes.flat, checkpoints):
    N1 = flips[:n].sum()
    N0 = n - N1
    a_post = a0 + N1
    b_post = b0 + N0

    prior_pdf   = beta_dist.pdf(theta_plot, a0, b0)
    post_pdf    = beta_dist.pdf(theta_plot, a_post, b_post)

    if n > 0:
        # [H4] DÜZELTİLDİ: gereksiz N1_arr değişkeni kaldırıldı
        # [H1] DÜZELTİLDİ: np.trapz → np.trapezoid (NumPy 2.0 uyumlu)
        lik = theta_plot ** N1 * (1 - theta_plot) ** N0
        lik /= np.trapezoid(lik, theta_plot)  # normalize
        ax.plot(theta_plot, lik, 'g-', lw=1.5, alpha=0.7, label='Olabilirlik')

    ax.plot(theta_plot, prior_pdf, '--', color='gray', lw=1.5, alpha=0.7, label=f'Önsel Beta({a0},{b0})')
    ax.plot(theta_plot, post_pdf, color=BLUE, lw=2.5, label=f'Sonsal Beta({a_post},{b_post})')
    ax.axvline(x=theta_true, color='red', ls=':', lw=2, alpha=0.8, label=f'Gerçek θ={theta_true}')

    mle  = N1/n if n > 0 else 0.5
    map_ = (a_post - 1)/(a_post + b_post - 2) if (a_post > 1 and b_post > 1) else 0.5
    mean = a_post / (a_post + b_post)

    info = f'N={n}, Y={N1}, T={N0}'
    if n > 0:
        info += f'\nMLE={mle:.2f}, MAP={map_:.2f}, Ort={mean:.2f}'
    ax.set_title(info, fontsize=10)
    ax.set_xlabel('θ'); ax.set_ylabel('Yoğunluk')
    ax.legend(fontsize=7)
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

---
### 2.4 MLE, MAP ve Sonsal Ortalama — Formüller ve Karşılaştırma

$$\hat{\theta}_{MLE} = \frac{N_1}{N}, \qquad
\hat{\theta}_{MAP} = \frac{N_1 + a - 1}{N + a + b - 2}, \qquad
\bar{\theta} = \frac{N_1 + a}{N + a + b}$$

Sonsal ortalama = öncel ağırlıklı MLE:
$$\mathbb{E}[\theta\mid\mathcal{D}] = \lambda m_1 + (1-\lambda)\hat{\theta}_{MLE}, \qquad \lambda=\frac{\alpha_0}{N+\alpha_0}$$

In [ ]:
# ── MLE / MAP / Sonsal Ortalama: N'e göre yakınsama ──────────────────────────
theta_true_est = 0.6
a, b = 3, 3          # Beta(3,3) önsel → öncel ortalama = 0.5
alpha0 = a + b       # etkin örneklem büyüklüğü
m1     = a / alpha0  # öncel ortalama

N_range = np.arange(1, 201)
np.random.seed(0)
all_flips = np.random.binomial(1, theta_true_est, 200)

mle_vals, map_vals, mean_vals, lambda_vals = [], [], [], []
for n in N_range:
    N1 = all_flips[:n].sum()
    N0 = n - N1
    mle_vals.append(N1 / n)
    ap, bp = a + N1, b + N0
    map_vals.append((ap - 1) / (ap + bp - 2) if ap > 1 and bp > 1 else 0.5)
    mean_vals.append(ap / (ap + bp))
    lambda_vals.append(alpha0 / (n + alpha0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MLE · MAP · Sonsal Ortalama — N Arttıkça Yakınsama', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(N_range, mle_vals,  '#E53935', lw=2, label='MLE')
ax.plot(N_range, map_vals,  '#FB8C00', lw=2, label='MAP')
ax.plot(N_range, mean_vals, '#1565c0', lw=2, label='Sonsal Ortalama')
ax.axhline(y=theta_true_est, color='black', ls='--', lw=1.5, alpha=0.7, label=f'Gerçek θ={theta_true_est}')
ax.axhline(y=m1, color='gray', ls=':', lw=1.5, label=f'Önsel ort. m₁={m1:.2f}')
ax.set_xlabel('N'); ax.set_ylabel('θ tahmini')
ax.set_title(f'Beta({a},{b}) önsel, gerçek θ={theta_true_est}')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(N_range, lambda_vals, BLUE, lw=2.5)
ax.fill_between(N_range, lambda_vals, alpha=0.2, color=BLUE)
ax.set_xlabel('N'); ax.set_ylabel('λ = α₀ / (N + α₀)')
ax.set_title('Önsel Ağırlığı λ: N arttıkça önsel azalır\nE[θ|D] = λ·m₁ + (1-λ)·θ̂_MLE')
ax.axhline(y=0.5, ls='--', color='gray', alpha=0.5)
ax.annotate(f'α₀ = {alpha0}', xy=(alpha0, 0.5), xytext=(alpha0+10, 0.55),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=11, color='red')

plt.tight_layout()
plt.show()

print('\n📊 N=5 için detay:')
n = 5; N1 = all_flips[:n].sum(); N0 = n - N1
ap, bp = a + N1, b + N0
lam = alpha0 / (n + alpha0)
mle = N1/n; mean = ap/(ap+bp)
print(f'   N₁={N1}, N₀={N0}')
print(f'   MLE = {mle:.3f}')
print(f'   Sonsal ort. = λ·m₁ + (1-λ)·MLE = {lam:.2f}·{m1:.2f} + {1-lam:.2f}·{mle:.2f} = {mean:.3f}')

---
### 2.5 Sıfır Sayım Problemi & Laplace Ardışıklık Kuralı

**Problem:** N=3 tura → MLE: θ̂ = 0 → yazı gelme ihtimali sıfır tahmin edilir!

**Çözüm (Laplace, Beta(1,1) önsel):**
$$p(\tilde{x}=1 \mid \mathcal{D}) = \frac{N_1 + 1}{N_1 + N_0 + 2}$$

In [ ]:
# ── Sıfır Sayım Problemi ───────────────────────────────────────────────────────
print('🦢 SIFIR SAYIM PROBLEMİ — Siyah Kuğu Paradoksu')
print('='*58)

experiments = [
    (0, 3,  'N=3 tura (0 yazı)'),
    (0, 10, 'N=10 tura (0 yazı)'),
    (3, 7,  'N=10, 3 yazı 7 tura'),
    (7, 3,  'N=10, 7 yazı 3 tura'),
]

for N1, N0, desc in experiments:
    N = N1 + N0
    mle       = N1 / N if N > 0 else 0
    laplace   = (N1 + 1) / (N + 2)        # Beta(1,1) önsel
    bayes_22  = (N1 + 2) / (N + 4)        # Beta(2,2) önsel
    danger    = '  ⚠️ Tehlikeli!' if mle == 0 else ''
    print(f'\n  {desc}')
    print(f'    MLE:             p(yazı) = {mle:.4f}{danger}')
    print(f'    Laplace:         p(yazı) = {laplace:.4f}')
    print(f'    Bayes Beta(2,2): p(yazı) = {bayes_22:.4f}')

# Görselleştirme: N₁=0 durumunda farklı önsel seçimlerinin etkisi
N0_vals = np.arange(1, 21)
fig, ax = plt.subplots(figsize=(10, 5))

for (a_lap, b_lap, label, color) in [
        (0, 0, 'MLE (a=0,b=0)',         '#E53935'),
        (1, 1, 'Laplace / Add-1 (a=b=1)','#1565c0'),
        (2, 2, 'Beta(2,2)',              '#2E7D32'),
        (5, 5, 'Beta(5,5) güçlü önsel', '#7B1FA2'),
]:
    preds = [(0 + a_lap) / (n0 + a_lap + b_lap) if (n0 + a_lap + b_lap) > 0 else 0
             for n0 in N0_vals]
    ax.plot(N0_vals, preds, 'o-', lw=2, ms=6, color=color, label=label)

ax.set_xlabel('N₀ (ardışık tura sayısı, N₁=0)', fontsize=12)
ax.set_ylabel('p(x̃=1 | D) — yazı tahmini', fontsize=12)
ax.set_title('Sıfır Sayım: MLE sıfıra yığılır, Bayesçi yöntemler düzeltir', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(-0.02, 0.55)
plt.tight_layout()
plt.show()

---
### 2.6 Sonsal Varyans ve Güven Aralığı

$$\text{var}[\theta \mid \mathcal{D}] \approx \frac{\hat{\theta}(1-\hat{\theta})}{N}, \qquad
\sigma \approx \sqrt{\frac{\hat{\theta}(1-\hat{\theta})}{N}}$$

In [ ]:
# ── Sonsal Varyans ve Güven Bantları ──────────────────────────────────────────
theta_true_var = 0.65
a_v, b_v = 2, 2
N_range_v = np.arange(5, 201, 5)
np.random.seed(42)
flips_v = np.random.binomial(1, theta_true_var, 200)

means_v, stds_v, exact_stds = [], [], []
for n in N_range_v:
    N1v = flips_v[:n].sum(); N0v = n - N1v
    ap = a_v + N1v; bp = b_v + N0v
    mean_v = ap / (ap + bp)
    # Tam Beta varyansı
    var_exact = (ap * bp) / ((ap + bp)**2 * (ap + bp + 1))
    # Yaklaşık varyans (N >> a,b)
    theta_hat = N1v / n
    var_approx = theta_hat * (1 - theta_hat) / n
    means_v.append(mean_v)
    stds_v.append(np.sqrt(var_approx))
    exact_stds.append(np.sqrt(var_exact))

means_v = np.array(means_v)
stds_v  = np.array(stds_v)
exact_stds = np.array(exact_stds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Sonsal Varyans ve Belirsizlik Azalması', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(N_range_v, means_v, BLUE, lw=2.5, label='Sonsal Ortalama')
ax.fill_between(N_range_v, means_v - 2*exact_stds, means_v + 2*exact_stds,
                alpha=0.2, color=BLUE, label='±2σ (tam varyans)')
ax.fill_between(N_range_v, means_v - stds_v, means_v + stds_v,
                alpha=0.35, color='orange', label='±1σ (yaklaşık)')
ax.axhline(y=theta_true_var, color='red', ls='--', lw=2, label=f'Gerçek θ={theta_true_var}')
ax.set_xlabel('N'); ax.set_ylabel('θ tahmini')
ax.set_title('N arttıkça güven aralığı daralır')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(N_range_v, exact_stds, BLUE, lw=2.5, label='Tam σ (Beta)')
ax.plot(N_range_v, stds_v,    'orange', lw=2, ls='--', label='Yaklaşık σ = √(θ̂(1-θ̂)/N)')
N_theory = np.linspace(5, 200, 300)
ax.plot(N_theory, np.sqrt(0.25 / N_theory), 'gray', ls=':', lw=1.5, label='Üst sınır: 0.5/√N')
ax.set_xlabel('N'); ax.set_ylabel('Sonsal Standart Sapma σ')
ax.set_title('Belirsizlik: σ ~ 1/√N oranında azalır')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('\n💡 Anahtar Nokta: Belirsizliği yarıya indirmek için 4× daha fazla veriye ihtiyaç var (σ ∝ 1/√N).')

---
### 2.7 Bileşik Beta-Binom Dağılımı — Gelecek Tahmini

Gelecekteki M denemede x yazı gelme olasılığı:

$$\text{Bb}(x \mid a, b, M) = \binom{M}{x}\frac{B(x+a,\,M-x+b)}{B(a,b)}$$

$$\mathbb{E}[x] = M\frac{a}{a+b}, \qquad \text{var}[x] = M\frac{ab}{(a+b)^2}\frac{a+b+M}{a+b+1}$$

In [ ]:
# ── Bileşik Beta-Binom ─────────────────────────────────────────────────────────
def beta_binom_pmf(x, a, b, M):
    """Bb(x|a,b,M) PMF — log-uzay ile sayısal kararlılık."""
    log_pmf = (gammaln(M + 1) - gammaln(x + 1) - gammaln(M - x + 1)
               + betaln(x + a, M - x + b) - betaln(a, b))
    return np.exp(log_pmf)

M = 20   # gelecekteki deneme sayısı

prior_configs = [
    (1,  1,  '#78909C', 'Beta(1,1) — Laplace (düzgün)'),
    (3,  7,  '#EF5350', 'Beta(3,7) — Turaya eğilim'),
    (7,  3,  '#42A5F5', 'Beta(7,3) — Yazıya eğilim'),
    (10, 10, '#66BB6A', 'Beta(10,10) — Dengeli güçlü'),
]

# Gözlemlenen veri: 6 yazı, 4 tura
N1_obs, N0_obs = 6, 4

x_vals = np.arange(0, M + 1)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(f'Bileşik Beta-Binom: Gelecekteki M={M} Denemede Yazı Sayısı Tahmini\n'
             f'(Gözlem: N₁={N1_obs} yazı, N₀={N0_obs} tura)',
             fontsize=13, fontweight='bold')

for ax, (a, b, color, label) in zip(axes.flat, prior_configs):
    # Sonsal parametreler
    a_post = a + N1_obs
    b_post = b + N0_obs

    pmf_bb    = np.array([beta_binom_pmf(x, a_post, b_post, M) for x in x_vals])
    pmf_binom = binom.pmf(x_vals, M, N1_obs / (N1_obs + N0_obs))  # MLE plug-in

    E_x   = M * a_post / (a_post + b_post)
    var_x = M * a_post * b_post / (a_post + b_post)**2 * (a_post + b_post + M) / (a_post + b_post + 1)

    ax.bar(x_vals - 0.2, pmf_bb,    0.4, color=color, alpha=0.8, label='Beta-Binom (Bayesçi)')
    ax.bar(x_vals + 0.2, pmf_binom, 0.4, color='gray', alpha=0.6, label='Binom/MLE (Tak-çıkar)')
    ax.axvline(x=E_x, color='red', ls='--', lw=1.8, label=f'E[x]={E_x:.1f}')
    ax.set_title(f'{label}\nSonsal: Beta({a_post},{b_post})  —  E[x]={E_x:.1f}, σ={np.sqrt(var_x):.1f}',
                 fontsize=10)
    ax.set_xlabel('x (yazı sayısı)'); ax.set_ylabel('Olasılık')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\n💡 Beta-Binom, Binom/MLE'ye kıyasla:")
print('   → Daha geniş (daha fazla belirsizlik içerir)')
print('   → Önsel bilgiyi hesaba katar')
print('   → Az veriyle daha güvenilir tahmin sağlar')

---
## 📌 BÖLÜM 3: Çok Kategorili Genelleme — Dirichlet-Multinomial
### 3.1 Multinomial Olabilirlik ve Dirichlet Önsel

K kategorili model için:
$$p(\mathcal{D}\mid\boldsymbol{\theta}) = \prod_{k=1}^K \theta_k^{N_k}$$

$$\text{Dir}(\boldsymbol{\theta}\mid\boldsymbol{\alpha}) = \frac{1}{B(\boldsymbol{\alpha})}\prod_{k=1}^K \theta_k^{\alpha_k - 1}$$

In [ ]:
# ── Dirichlet Dağılımı: K=3 Simplex Görselleştirmesi ─────────────────────────
def sample_dirichlet_simplex(alpha, n_samples=5000):
    """Dirichlet örneklerini eşkenar üçgen üzerine barycentric koordinatla projekte eder.
    Köşeler: A=(0,0): θ₁=1, B=(1,0): θ₂=1, C=(0.5, √3/2): θ₃=1
    """
    samples = np.random.dirichlet(alpha, n_samples)
    # [H8] DÜZELTİLDİ: samples.sum(1) her zaman 1.0 — gereksiz bölme kaldırıldı
    # Doğru barycentric koordinatlar:
    x = samples[:, 1] + samples[:, 2] * 0.5
    y = samples[:, 2] * (np.sqrt(3) / 2)
    return x, y

alpha_configs = [
    ([1,   1,   1  ], 'Dir(1,1,1) — Düzgün (Laplace)'),
    ([5,   5,   5  ], 'Dir(5,5,5) — Dengeli yoğun'),
    ([0.5, 0.5, 0.5], 'Dir(0.5,0.5,0.5) — Köşelerde (sparse)'),
    ([5,   2,   1  ], 'Dir(5,2,1) — θ₁ baskın'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Dirichlet Dağılımı — K=3 için Olasılık Simpleksi', fontsize=14, fontweight='bold')

for ax, (alpha, title) in zip(axes.flat, alpha_configs):
    np.random.seed(0)
    x, y = sample_dirichlet_simplex(alpha)
    ax.scatter(x, y, s=3, alpha=0.3, color=BLUE)
    # Üçgen sınırı
    tri_x = [0, 1, 0.5, 0]; tri_y = [0, 0, np.sqrt(3)/2, 0]
    ax.plot(tri_x, tri_y, 'k-', lw=1.5)
    ax.text(-0.05, -0.05, 'θ₁=1', fontsize=10, ha='center')
    ax.text(1.05,  -0.05, 'θ₂=1', fontsize=10, ha='center')
    ax.text(0.5,  np.sqrt(3)/2 + 0.04, 'θ₃=1', fontsize=10, ha='center')
    ax.set_title(title, fontsize=11)
    ax.set_aspect('equal'); ax.axis('off')

plt.tight_layout()
plt.show()

---
### 3.2 Dirichlet-Multinomial Güncelleme

$$p(\boldsymbol{\theta}\mid\mathcal{D}) = \text{Dir}(\boldsymbol{\theta}\mid\alpha_1+N_1,\ldots,\alpha_K+N_K)$$

**Sonsal Tahmin (tek deneme):**
$$p(X=j\mid\mathcal{D}) = \frac{\alpha_j + N_j}{\alpha_0 + N}$$

In [ ]:
# ── Zar Atışı: Dirichlet-Multinomial ─────────────────────────────────────────
np.random.seed(42)
K = 6   # Adil bir zar
theta_true_die = np.array([0.1, 0.1, 0.1, 0.2, 0.2, 0.3])  # hileli zar
theta_true_die /= theta_true_die.sum()

alpha_prior = np.ones(K)   # Dir(1,...,1): düzgün önsel
alpha0 = alpha_prior.sum()

N_total_die = 100
rolls = np.random.choice(K, size=N_total_die, p=theta_true_die)
counts = np.bincount(rolls, minlength=K)

# Bayesçi güncelleme adımları
checkpoints_die = [0, 5, 20, 50, 100]
fig, axes = plt.subplots(1, len(checkpoints_die), figsize=(16, 5), sharey=True)
fig.suptitle('Dirichlet-Multinomial: Zar Olasılıklarını Öğrenme (Hileli Zar)',
             fontsize=13, fontweight='bold')

labels = [f'Yüz {i+1}' for i in range(K)]

for ax, n in zip(axes, checkpoints_die):
    cnt = np.bincount(rolls[:n], minlength=K)
    alpha_post = alpha_prior + cnt
    N_total_n  = cnt.sum()

    # MLE (ya da uniform prior durumu)
    mle_est = cnt / N_total_n if N_total_n > 0 else np.ones(K)/K
    # Sonsal ortalama
    post_mean = alpha_post / alpha_post.sum()

    x_pos = np.arange(K)
    ax.bar(x_pos - 0.2, theta_true_die, 0.2, color='black', alpha=0.4, label='Gerçek θ')
    ax.bar(x_pos,       mle_est,        0.2, color='orange', alpha=0.8, label='MLE')
    ax.bar(x_pos + 0.2, post_mean,      0.2, color=BLUE,    alpha=0.8, label='Sonsal ort.')

    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, rotation=45, fontsize=8)
    ax.set_title(f'N = {n}', fontsize=11)
    ax.set_ylabel('Olasılık')
    if n == checkpoints_die[0]:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Sayısal özet
print('\n📊 Tahmin Özeti (N=100):')
alpha_post_final = alpha_prior + counts
post_mean_final  = alpha_post_final / alpha_post_final.sum()
mle_final        = counts / counts.sum()

# [H2] DÜZELTİLDİ: f-string dış sarmalayıcı çift tırnak yapıldı
print(f"  {'Yüz':>6}  {'Gerçek':>8}  {'MLE':>8}  {'Sonsal Ort.':>12}  {'Hata(MLE)':>10}  {'Hata(Bayes)':>12}")
print('-'*65)
for i in range(K):
    print(f'  {i+1:>6}  {theta_true_die[i]:>8.4f}  {mle_final[i]:>8.4f}  '
          f'{post_mean_final[i]:>12.4f}  '
          f'{abs(mle_final[i]-theta_true_die[i]):>10.4f}  '
          f'{abs(post_mean_final[i]-theta_true_die[i]):>12.4f}')

---
### 3.3 Add-K Düzeltme — Dirichlet Laplace

β hiper-parametresiyle **add-k düzeltmesi:**
$$p(X=j\mid\mathcal{D}) = \frac{N_j + \beta}{N + K\beta}$$

- β = 1: Laplace (add-one)
- β → 0: MLE
- β → ∞: Uniform tahmin

In [ ]:
# ── Add-K Düzeltmesi: Metin Sınıflandırma Benzetimi ──────────────────────────
# Küçük bir kelime sayımı senaryosu
vocab = ['yapay', 'zeka', 'model', 'veri', 'öğrenme', 'binom', 'nadir_kelime']
K_vocab = len(vocab)

# Gerçek kelime frekansları
true_freq = np.array([0.30, 0.25, 0.20, 0.15, 0.05, 0.04, 0.01])

# Küçük gözlem kümesi (bazı kelimeler hiç görülmemiş!)
observed_counts = np.array([15, 12, 9, 7, 2, 0, 0])
N_obs = observed_counts.sum()

beta_vals = [0.0001, 0.5, 1.0, 5.0]
labels_b  = ['MLE (β≈0)', 'Add-0.5', 'Laplace (β=1)', 'Strong prior (β=5)']
colors_b  = ['#E53935', '#FB8C00', '#1565c0', '#388E3C']

fig, ax = plt.subplots(figsize=(12, 6))
x_vocab = np.arange(K_vocab)
width   = 0.15

ax.bar(x_vocab - 1.5*width, true_freq, width, color='black', alpha=0.5, label='Gerçek dağılım')
for i, (beta, label, color) in enumerate(zip(beta_vals, labels_b, colors_b)):
    preds = (observed_counts + beta) / (N_obs + K_vocab * beta)
    ax.bar(x_vocab + (i - 0.5)*width, preds, width, color=color, alpha=0.8, label=label)

ax.set_xticks(x_vocab)
ax.set_xticklabels(vocab, rotation=30, ha='right', fontsize=11)
ax.set_ylabel('Olasılık Tahmini')
ax.set_title('Add-K Düzeltmesi: Nadir Kelimeler İçin Olasılık Tahminleri\n'
             '("nadir_kelime" ve "binom" gözlemlenmedi — MLE = 0!)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 0.40)
plt.tight_layout()
plt.show()

print('\n📊 Add-K Sonuçları (nadir_kelime için):')
# [H2] DÜZELTİLDİ: f-string dış sarmalayıcı çift tırnak yapıldı
print(f"  {'Yöntem':20}  {'p(nadir_kelime)'}")
print('-' * 40)
for beta, label in zip(beta_vals, labels_b):
    p = (0 + beta) / (N_obs + K_vocab * beta)
    warning = '  ← SIFIR!' if beta < 0.001 else ''
    print(f'  {label:20}  {p:.6f}{warning}')

---
## 📌 BÖLÜM 4: Karışım Modeli — İnsan Sezgisinin Modellenmesi

$$p(h) = \pi_0\, p_{rules}(h) + (1-\pi_0)\, p_{interval}(h)$$

π₀ → ne kadar "matematikçi" (kurallara dayalı) mi, ne kadar "istatistikçi" (aralık-tabanlı) mi?

In [ ]:
# ── π₀'ın Sonsal Öngörüsel Dağılıma Etkisi ───────────────────────────────────
data_mix = [16, 8, 2, 64]
pi0_values = [0.1, 0.5, 0.9, 0.99]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(f'Karışım Önseli: π₀ Parametresinin Etkisi  (D = {data_mix})\n'
             'π₀ büyüdükçe model "matematiksel kurallara" daha çok güvenir',
             fontsize=13, fontweight='bold')

for ax, pi0 in zip(axes.flat, pi0_values):
    prior_mix = build_prior(hypotheses, pi0=pi0)
    post_mix  = compute_posterior(data_mix, hypotheses, prior_mix)
    preds_mix = np.array([posterior_predictive(x, post_mix, hypotheses) for x in xs])
    if preds_mix.max() > 0:
        preds_mix /= preds_mix.max()

    # TOP-3 hipotezi bul
    top3 = sorted(post_mix.items(), key=lambda x: x[1], reverse=True)[:3]

    ax.bar(xs, preds_mix, color=BLUE, alpha=0.8, width=0.8)
    for d in data_mix:
        ax.axvline(x=d, color='red', lw=1.5, alpha=0.8)
    ax.set_title(f'π₀ = {pi0}  (kurallar ağırlığı)\n'
                 + '  '.join([f'{h[:12]}:{p:.2f}' for h,p in top3]),
                 fontsize=10)
    ax.set_xlim(0, 101); ax.set_ylim(0, 1.2)
    ax.set_xticks(range(0, 101, 10))
    ax.set_ylabel('p(x̃ ∈ C|D)')
    ax.set_xlabel('x̃')

plt.tight_layout()
plt.show()

print('\n💡 Gözlem:')
print("  π₀ büyük (0.9-0.99): Model 2'nin kuvvetleri gibi keskin kurallara yönelir")
print('  π₀ küçük (0.1):      Model aralık hipotezlerine daha fazla ağırlık verir')
print('  İnsanlar genellikle π₀ > 0.5 ile modellenir (kural tabanlı düşünce baskın)')

---
## 📌 BÖLÜM 5: Kapsamlı Karşılaştırma — MLE vs MAP vs Bayes


In [ ]:
# ── 5.1 Üç Yöntemin Özet Karşılaştırması ─────────────────────────────────────
np.random.seed(1)
theta_compare = 0.7
a_c, b_c = 2, 5   # Turaya eğilimli önsel (θ=0.5 civarı)

n_experiments = [3, 5, 10, 20, 50, 100, 500]
results = {'MLE': [], 'MAP': [], 'Sonsal Ort.': [], 'N': n_experiments}

all_flips_cmp = np.random.binomial(1, theta_compare, 500)

for n in n_experiments:
    N1c = all_flips_cmp[:n].sum()
    N0c = n - N1c
    ap, bp = a_c + N1c, b_c + N0c

    results['MLE'].append(N1c / n)
    results['MAP'].append((ap - 1) / (ap + bp - 2))
    results['Sonsal Ort.'].append(ap / (ap + bp))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'MLE vs MAP vs Bayesçi Ortalama  (gerçek θ={theta_compare}, önsel Beta({a_c},{b_c}))',
             fontsize=13, fontweight='bold')

ax = axes[0]
for method, color, ls in [('MLE','#E53935','-'), ('MAP','#FB8C00','--'), ('Sonsal Ort.',BLUE,'-')]:
    ax.plot(n_experiments, results[method], color=color, lw=2.5, ls=ls,
            marker='o', ms=6, label=method)
ax.axhline(y=theta_compare, color='black', ls=':', lw=1.5, label=f'Gerçek θ={theta_compare}')
ax.axhline(y=a_c/(a_c+b_c), color='gray', ls='-.', lw=1.5, label=f'Önsel ort.={a_c/(a_c+b_c):.2f}')
ax.set_xscale('log'); ax.set_xlabel('N (log ölçek)'); ax.set_ylabel('θ tahmini')
ax.set_title('θ Tahminleri vs N'); ax.legend(fontsize=9)

# Hata karşılaştırması
ax = axes[1]
for method, color, ls in [('MLE','#E53935','-'), ('MAP','#FB8C00','--'), ('Sonsal Ort.',BLUE,'-')]:
    errors = [abs(v - theta_compare) for v in results[method]]
    ax.plot(n_experiments, errors, color=color, lw=2.5, ls=ls,
            marker='s', ms=6, label=f'{method} hatası')
ax.set_xscale('log'); ax.set_xlabel('N (log ölçek)'); ax.set_ylabel('|tahmin - gerçek|')
ax.set_title('Mutlak Hata vs N'); ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# [H2] DÜZELTİLDİ: f-string dış sarmalayıcı çift tırnak yapıldı
print('\n📊 Özet Tablo:')
print(f"  {'N':>6}  {'MLE':>8}  {'MAP':>8}  {'Bayes':>8}")
print('-' * 36)
for i, n in enumerate(n_experiments):
    print(f"  {n:>6}  {results['MLE'][i]:>8.4f}  {results['MAP'][i]:>8.4f}  {results['Sonsal Ort.'][i]:>8.4f}")
print(f'  {"Gerçek":>6}  {theta_compare:>8.4f}  {theta_compare:>8.4f}  {theta_compare:>8.4f}')

In [ ]:
# ── 5.2 Ders Özeti: Anahtar Formüller ────────────────────────────────────────
print('=' * 65)
print('  📚 DERS 2 ÖZET — Anahtar Formüller ve Kavramlar')
print('=' * 65)

sections = [
    ('Number Game / Kavram Öğrenme', [
        ('Güçlü Örnekleme (Olabilirlik)', 'p(D|h) = (1/|h|)^N'),
        ('Sonsal',                         'p(h|D) ∝ p(D|h)·p(h)'),
        ('BMA',                            'p(x̃∈C|D) = Σ_h p(y=1|x̃,h)·p(h|D)'),
        ('Tak-çıkar',                      'p(x̃∈C|D) ≈ p(x̃|ĥ_MAP)'),
        ('MAP (N→∞)',                      'p(h|D) → δ_{ĥ_MAP}(h)'),
        ('Karışım Önseli',                 'p(h) = π₀·p_rules(h) + (1-π₀)·p_interval(h)'),
    ]),
    ('Beta-Binom (İkili Durum)', [
        ('Bernoulli Olabilirlik',      'p(D|θ) = θ^N₁·(1-θ)^N₀'),
        ('Beta Önsel',                 'p(θ) = Beta(θ|a,b)'),
        ('Beta Sonsal',                'p(θ|D) = Beta(θ|N₁+a, N₀+b)'),
        ('MLE',                        'θ̂_MLE = N₁/N'),
        ('MAP',                        'θ̂_MAP = (N₁+a-1)/(N+a+b-2)'),
        ('Sonsal Ortalama',            'E[θ|D] = (N₁+a)/(N+a+b)'),
        ('Laplace Ardışıklık Kuralı',  'p(x̃=1|D) = (N₁+1)/(N+2)'),
        ('Bileşik Beta-Binom',         'Bb(x|a,b,M) = C(M,x)·B(x+a,M-x+b)/B(a,b)'),
    ]),
    ('Dirichlet-Multinomial (K Kategori)', [
        ('Multinomial Olabilirlik', 'p(D|θ) = ∏_k θ_k^N_k'),
        ('Dirichlet Önsel',        'Dir(θ|α) = (1/B(α))·∏_k θ_k^{α_k-1}'),
        ('Dirichlet Sonsal',       'p(θ|D) = Dir(θ|α₁+N₁, …, α_K+N_K)'),
        ('Sonsal Tahmin',          'p(X=j|D) = (α_j+N_j)/(α₀+N)'),
        ('Add-K Düzeltmesi',       'p(X=j|D) = (N_j+β)/(N+K·β)'),
    ]),
]

for sec_title, formulas in sections:
    print(f'\n  ▌ {sec_title}')
    print('  ' + '─'*60)
    for i, (name, formula) in enumerate(formulas, 1):
        print(f'    {i}. {name}')
        print(f'       {formula}')

print('\n' + '=' * 65)
print('  ✅ Notebook tamamlandı!')
print('=' * 65)

---
## 🎯 Alıştırma Soruları

1. **Number Game:** D = {10, 20, 30} gözlemlediğinizde sonsal öngörüsel dağılımı hesaplayınız. Hangi hipotez baskın çıkıyor? D = {10, 20, 30, 40, 50} için tekrarlayınız.

2. **Boyut Prensibi:** |h₁| = 10, |h₂| = 50 için N = 1, 2, 5, 10 gözlem sayısında olabilirlik oranı p(D|h₁)/p(D|h₂)'yi hesaplayınız. N ne zaman baskın hale geliyor?

3. **Beta-Binom:** a=1, b=1 (Laplace) önseliyle başlayarak YTTYTYYYT dizisini adım adım güncelleyiniz. Her adımda MAP, MLE ve sonsal ortalamayı karşılaştırınız.

4. **Sıfır Sayım:** N=5 gözlemde hiç yazı gelmedi. β=0, 0.5, 1, 2 için p(x̃=1|D)'yi hesaplayınız. Hangisi daha makul?

5. **Dirichlet:** 6 yüzlü zarı 30 kez atarak, K=6 için Dirichlet sonsal dağılımını güncelleyiniz. Adil zar (θ_k = 1/6) hipotezi üzerindeki sonsal olasılığı yorumlayınız.

6. **Karışım Önseli:** π₀ = 0.3 ve π₀ = 0.95 için D={16} gözlemiyle sonsal öngörüsel dağılımı karşılaştırınız. İki durumdaki farkı yorumlayınız.

---
*Üretken Yapay Zeka — Haydar Kılıç, Yapay Zeka Mühendisliği*